<a href="https://colab.research.google.com/github/harshita10sharma/Youtube-video-dubbing-system/blob/main/Youtube_Dubbing_IndicTrans2_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q transformers sentencepiece torch IndicTransToolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.1/546.1 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import transformers
import sentencepiece
from IndicTransToolkit.processor import IndicProcessor

print("Transformers:", transformers.__version__)
print("IndicTransToolkit: OK")
print("SentencePiece: OK")

Transformers: 4.53.2
IndicTransToolkit: OK
SentencePiece: OK


In [6]:
from huggingface_hub import login

login()

In [7]:
from huggingface_hub import whoami

info = whoami()
print("Logged in as:", info["name"])

Logged in as: harshita10sh


In [8]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "ai4bharat/indictrans2-indic-en-dist-200M"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading IndicTrans2...")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
).to(device)

print("IndicTrans2 loaded successfully!")

Loading IndicTrans2...
Device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

IndicTrans2 loaded successfully!


In [9]:
from IndicTransToolkit.processor import IndicProcessor

ip = IndicProcessor(inference=True)

source_text = [
    "नमस्ते, मेरा नाम हर्षिता है।",
    "मैं मशीन लर्निंग और आर्टिफिशियल इंटेलिजेंस पर काम कर रही हूँ।",
    "यह एक वीडियो डबिंग सिस्टम है।"
]

# IndicTrans2 language codes
src_lang = "hin_Deva"
tgt_lang = "eng_Latn"

# Prepare input
batch = ip.preprocess_batch(
    source_text,
    src_lang=src_lang,
    tgt_lang=tgt_lang
)

# Tokenize
inputs = tokenizer(
    batch,
    padding=True,
    truncation=True,
    return_tensors="pt"
).to(device)

# Generate translation
with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        num_beams=5,
        num_return_sequences=1,
        max_length=256
    )

# Decode
generated_text = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)

# Post-process
translations = ip.postprocess_batch(
    generated_text,
    lang=tgt_lang
)

print("\nHindi → English\n")
for src, tgt in zip(source_text, translations):
    print("Hindi :", src)
    print("English:", tgt)
    print()


Hindi → English

Hindi : नमस्ते, मेरा नाम हर्षिता है।
English: Hi, my name is Harshita.

Hindi : मैं मशीन लर्निंग और आर्टिफिशियल इंटेलिजेंस पर काम कर रही हूँ।
English: I am working on machine learning and artificial intelligence.

Hindi : यह एक वीडियो डबिंग सिस्टम है।
English: It is a video dubbing system.



In [12]:
import json

with open("transcript.json", "r", encoding="utf-8") as f:
    transcript = json.load(f)

print("Number of Whisper segments:", len(transcript))

print("\nFirst 5 segments:\n")

for i, segment in enumerate(transcript[:5]):
    print(f"[{i}]")
    print("Start :", segment["start"])
    print("End   :", segment["end"])
    print("Text  :", segment["text"])
    print()

Number of Whisper segments: 260

First 5 segments:

[0]
Start : 10.06
End   : 14.06
Text  : कहानिया हम सब की जिन्दिगी में बहुत महतुपून होती है, बहुत इंपूरेंट होती हैं।

[1]
Start : 14.06
End   : 16.06
Text  : यह बात आप समझते हैं, जानते हैं।

[2]
Start : 16.06
End   : 19.06
Text  : आप सोचीए वो सारी कहानिया जो हमने बचपं में स्कूलो में पढडली,

[3]
Start : 19.06
End   : 21.06
Text  : हमारे माबाप नहीं हमें सुना दी।

[4]
Start : 21.06
End   : 23.06
Text  : हमारे दोस्तों से हमने कुछ सीखली



In [13]:
# Translate a small batch of the REAL Whisper transcript
# using IndicTrans2.

test_segments = transcript[:10]

source_text = [
    segment["text"].strip()
    for segment in test_segments
]

src_lang = "hin_Deva"
tgt_lang = "eng_Latn"

# Prepare the Hindi text for IndicTrans2
batch = ip.preprocess_batch(
    source_text,
    src_lang=src_lang,
    tgt_lang=tgt_lang
)

# Tokenize
inputs = tokenizer(
    batch,
    padding=True,
    truncation=True,
    return_tensors="pt"
).to(device)

# Generate translations
with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        num_beams=5,
        num_return_sequences=1,
        max_length=256
    )

# Decode
generated_text = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)

# Post-process
translations = ip.postprocess_batch(
    generated_text,
    lang=tgt_lang
)

# Display original + translation + timestamps
print("Hindi → English\n")

for segment, translation in zip(test_segments, translations):

    print(
        f"[{segment['start']:.2f}s - {segment['end']:.2f}s]"
    )

    print("Hindi   :", segment["text"])
    print("English :", translation)
    print()

Hindi → English

[10.06s - 14.06s]
Hindi   : कहानिया हम सब की जिन्दिगी में बहुत महतुपून होती है, बहुत इंपूरेंट होती हैं।
English : Stories are very important, very important in the life of all of us.

[14.06s - 16.06s]
Hindi   : यह बात आप समझते हैं, जानते हैं।
English : You know that, you know that.

[16.06s - 19.06s]
Hindi   : आप सोचीए वो सारी कहानिया जो हमने बचपं में स्कूलो में पढडली,
English : Think of all the stories we read in school as children.

[19.06s - 21.06s]
Hindi   : हमारे माबाप नहीं हमें सुना दी।
English : Our parents didn't listen to us.

[21.06s - 23.06s]
Hindi   : हमारे दोस्तों से हमने कुछ सीखली
English : We learned something from our friends.

[23.06s - 27.06s]
Hindi   : वो सारी कहानियों का अमाल्गमेशन है शक्स जो मेरे सामने बैठा है
English : It's an amalgamation of all the stories that Shaq is sitting in front of me.

[27.06s - 29.06s]
Hindi   : उनमेसे एक भी कहानि अगर नहीं सुनाई गय होती
English : If not one of them had been heard.

[29.06s - 32.06s]
Hindi   : तो आज आप 

In [15]:
def merge_short_segments(
    segments,
    max_gap=0.6,
    max_duration=12.0,
):
    """
    Merge consecutive Whisper segments into larger
    sentence/context-level chunks.
    """

    if not segments:
        return segments

    merged = [dict(segments[0])]

    for seg in segments[1:]:
        last = merged[-1]

        gap = seg["start"] - last["end"]
        combined_duration = seg["end"] - last["start"]

        if gap <= max_gap and combined_duration <= max_duration:
            last["text"] = (
                last["text"].rstrip()
                + " "
                + seg["text"].lstrip()
            ).strip()

            last["end"] = seg["end"]

        else:
            merged.append(dict(seg))

    return merged


merged_transcript = merge_short_segments(transcript)

print("Original Whisper segments :", len(transcript))
print("Merged translation chunks :", len(merged_transcript))

print("\nFirst 10 merged chunks:\n")

for i, segment in enumerate(merged_transcript[:10]):
    print(f"[{i}] {segment['start']:.2f}s - {segment['end']:.2f}s")
    print(segment["text"])
    print()

Original Whisper segments : 260
Merged translation chunks : 90

First 10 merged chunks:

[0] 10.06s - 21.06s
कहानिया हम सब की जिन्दिगी में बहुत महतुपून होती है, बहुत इंपूरेंट होती हैं। यह बात आप समझते हैं, जानते हैं। आप सोचीए वो सारी कहानिया जो हमने बचपं में स्कूलो में पढडली, हमारे माबाप नहीं हमें सुना दी।

[1] 21.06s - 32.06s
हमारे दोस्तों से हमने कुछ सीखली वो सारी कहानियों का अमाल्गमेशन है शक्स जो मेरे सामने बैठा है उनमेसे एक भी कहानि अगर नहीं सुनाई गय होती तो आज आप जो पस्नालिटी है वो नहीं होते

[2] 32.06s - 44.06s
कुछ अलग होते कहानिया हम सब के लिए इतनी ज़रूरी इसले होती हैं. There are 7,139 languages in the world. I think there should be one more. The language of storytelling.

[3] 44.06s - 55.06s
बात करने का बात्छीत करने का सब से बहतरीन तरीका. Imagine कीजे. Imagine कीजे कि आप एक जहाज में हैं. पूरा अंधेरा हैं.

[4] 55.06s - 67.06s
हर तरफ तूफान आ रहा हैं. बहुत बढ़ी बढ़ी लहरें उठ रही हैं और सुड़िनली कहीं दूर आपको एक रौशनी दिकहें तो कैसे हूसला सा बढ़ जाता है सुकून सा आजाता है

[5] 67.06

In [16]:
# Translate the first 10 merged chunks using IndicTrans2

test_segments = merged_transcript[:10]

source_text = [
    segment["text"].strip()
    for segment in test_segments
]

src_lang = "hin_Deva"
tgt_lang = "eng_Latn"

# Prepare input
batch = ip.preprocess_batch(
    source_text,
    src_lang=src_lang,
    tgt_lang=tgt_lang
)

# Tokenize
inputs = tokenizer(
    batch,
    padding=True,
    truncation=True,
    return_tensors="pt"
).to(device)

# Generate translations
with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        num_beams=5,
        num_return_sequences=1,
        max_length=256
    )

# Decode
generated_text = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)

# Post-process
translations = ip.postprocess_batch(
    generated_text,
    lang=tgt_lang
)

# Display results
print("IndicTrans2 — Merged Hindi → English\n")

for segment, translation in zip(test_segments, translations):

    print(
        f"[{segment['start']:.2f}s - {segment['end']:.2f}s]"
    )
    print("Hindi   :", segment["text"])
    print("English :", translation)
    print("-" * 80)

IndicTrans2 — Merged Hindi → English

[10.06s - 21.06s]
Hindi   : कहानिया हम सब की जिन्दिगी में बहुत महतुपून होती है, बहुत इंपूरेंट होती हैं। यह बात आप समझते हैं, जानते हैं। आप सोचीए वो सारी कहानिया जो हमने बचपं में स्कूलो में पढडली, हमारे माबाप नहीं हमें सुना दी।
English : Stories are very important, very important in all our lives. You understand, you know. You think of all the stories that we read in school as children, not our parents told us.
--------------------------------------------------------------------------------
[21.06s - 32.06s]
Hindi   : हमारे दोस्तों से हमने कुछ सीखली वो सारी कहानियों का अमाल्गमेशन है शक्स जो मेरे सामने बैठा है उनमेसे एक भी कहानि अगर नहीं सुनाई गय होती तो आज आप जो पस्नालिटी है वो नहीं होते
English : Something we learned from our friends is the amalgamation of all the stories that are sitting in front of me, if not one of them had been heard, then you would not be the personality that you are today.
-----------------------------------------------------

In [18]:
# Languages currently supported by IndicTrans2
# for Indic -> English translation.

INDIC_LANGUAGES = {
    "as",   # Assamese
    "bn",   # Bengali
    "brx",  # Bodo
    "doi",  # Dogri
    "gom",  # Konkani
    "gu",   # Gujarati
    "hi",   # Hindi
    "kn",   # Kannada
    "ks",   # Kashmiri
    "mai",  # Maithili
    "ml",   # Malayalam
    "mr",   # Marathi
    "mni",  # Manipuri
    "ne",   # Nepali
    "or",   # Odia
    "pa",   # Punjabi
    "sa",   # Sanskrit
    "sat",  # Santali
    "sd",   # Sindhi
    "ta",   # Tamil
    "te",   # Telugu
    "ur",   # Urdu
}

def is_indic_language(language):
    return language in INDIC_LANGUAGES


test_languages = ["hi", "ta", "te", "bn", "fr", "de", "es"]

for language in test_languages:
    if is_indic_language(language):
        print(language, "→ IndicTrans2")
    else:
        print(language, "→ General translator")

hi → IndicTrans2
ta → IndicTrans2
te → IndicTrans2
bn → IndicTrans2
fr → General translator
de → General translator
es → General translator


In [19]:
import json
import torch

# The current test video is Hindi.
# In the final project this will come from Whisper's language detection.
whisper_language = "hi"

# Map Whisper language codes to IndicTrans2 language codes.
INDIC_LANGUAGE_CODES = {
    "as": "asm_Beng",
    "bn": "ben_Beng",
    "brx": "brx_Deva",
    "doi": "doi_Deva",
    "gom": "gom_Deva",
    "gu": "guj_Gujr",
    "hi": "hin_Deva",
    "kn": "kan_Knda",
    "ks": "kas_Arab",
    "mai": "mai_Deva",
    "ml": "mal_Mlym",
    "mr": "mar_Deva",
    "mni": "mni_Mtei",
    "ne": "npi_Deva",
    "or": "ory_Orya",
    "pa": "pan_Guru",
    "sa": "san_Deva",
    "sat": "sat_Olck",
    "sd": "snd_Arab",
    "ta": "tam_Taml",
    "te": "tel_Telu",
    "ur": "urd_Arab",
}

src_lang = INDIC_LANGUAGE_CODES[whisper_language]
tgt_lang = "eng_Latn"

print("Source language :", whisper_language)
print("IndicTrans2 code:", src_lang)
print("Target language :", tgt_lang)
print("Translation chunks:", len(merged_transcript))


def translate_batch(texts):
    """Translate one batch using IndicTrans2."""

    batch = ip.preprocess_batch(
        texts,
        src_lang=src_lang,
        tgt_lang=tgt_lang
    )

    inputs = tokenizer(
        batch,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            num_beams=5,
            num_return_sequences=1,
            max_length=256
        )

    generated_text = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return ip.postprocess_batch(
        generated_text,
        lang=tgt_lang
    )


# Translate in small batches.
BATCH_SIZE = 8

translated_segments = []

for start in range(0, len(merged_transcript), BATCH_SIZE):

    batch_segments = merged_transcript[
        start:start + BATCH_SIZE
    ]

    texts = [
        segment["text"].strip()
        for segment in batch_segments
    ]

    translations = translate_batch(texts)

    for segment, translation in zip(
        batch_segments,
        translations
    ):
        result = dict(segment)
        result["translated"] = translation
        translated_segments.append(result)

    print(
        f"Translated "
        f"{min(start + BATCH_SIZE, len(merged_transcript))}"
        f"/{len(merged_transcript)} chunks"
    )


print("\nTranslation complete!")

Source language : hi
IndicTrans2 code: hin_Deva
Target language : eng_Latn
Translation chunks: 90
Translated 8/90 chunks
Translated 16/90 chunks
Translated 24/90 chunks
Translated 32/90 chunks
Translated 40/90 chunks
Translated 48/90 chunks
Translated 56/90 chunks
Translated 64/90 chunks
Translated 72/90 chunks
Translated 80/90 chunks
Translated 88/90 chunks
Translated 90/90 chunks

Translation complete!


In [20]:
with open(
    "translated_indic.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        translated_segments,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved: translated_indic.json")
print("Total translated chunks:", len(translated_segments))

Saved: translated_indic.json
Total translated chunks: 90


In [21]:
for segment in translated_segments[:10]:

    print(
        f"[{segment['start']:.2f}s - "
        f"{segment['end']:.2f}s]"
    )

    print("Hindi   :", segment["text"])
    print("English :", segment["translated"])
    print("-" * 80)

[10.06s - 21.06s]
Hindi   : कहानिया हम सब की जिन्दिगी में बहुत महतुपून होती है, बहुत इंपूरेंट होती हैं। यह बात आप समझते हैं, जानते हैं। आप सोचीए वो सारी कहानिया जो हमने बचपं में स्कूलो में पढडली, हमारे माबाप नहीं हमें सुना दी।
English : Stories are very important, very important in all our lives. You understand, you know. You think of all the stories that we read in school as children, not our parents told us.
--------------------------------------------------------------------------------
[21.06s - 32.06s]
Hindi   : हमारे दोस्तों से हमने कुछ सीखली वो सारी कहानियों का अमाल्गमेशन है शक्स जो मेरे सामने बैठा है उनमेसे एक भी कहानि अगर नहीं सुनाई गय होती तो आज आप जो पस्नालिटी है वो नहीं होते
English : Something we learned from our friends is the amalgamation of all the stories that are sitting in front of me, if not one of them had been heard, then you would not be the personality that you are today.
--------------------------------------------------------------------------------
[32.06s - 

In [22]:
import sys
sys.path.append("/content")

from indic_translator import IndicTranslator

print("IndicTranslator imported successfully!")

IndicTranslator imported successfully!


In [23]:
translator = IndicTranslator()

print("Device:", translator.device)


Device: cuda


In [24]:
test_segments = merged_transcript[:10]

translated_test = translator.translate_segments(
    test_segments,
    source_language="hi",
    batch_size=8,
)

print("\nTranslation completed!")
print("Segments translated:", len(translated_test))

Loading IndicTrans2 model on cuda...
IndicTrans2 model loaded successfully.
Translated 8/10 segments
Translated 10/10 segments

Translation completed!
Segments translated: 10


In [25]:
for segment in translated_test:
    print(f"\n[{segment['start']:.2f}s - {segment['end']:.2f}s]")
    print("Hindi   :", segment["text"])
    print("English :", segment["translated"])


[10.06s - 21.06s]
Hindi   : कहानिया हम सब की जिन्दिगी में बहुत महतुपून होती है, बहुत इंपूरेंट होती हैं। यह बात आप समझते हैं, जानते हैं। आप सोचीए वो सारी कहानिया जो हमने बचपं में स्कूलो में पढडली, हमारे माबाप नहीं हमें सुना दी।
English : Stories are very important, very important in all our lives. You understand, you know. You think of all the stories that we read in school as children, not our parents told us.

[21.06s - 32.06s]
Hindi   : हमारे दोस्तों से हमने कुछ सीखली वो सारी कहानियों का अमाल्गमेशन है शक्स जो मेरे सामने बैठा है उनमेसे एक भी कहानि अगर नहीं सुनाई गय होती तो आज आप जो पस्नालिटी है वो नहीं होते
English : Something we learned from our friends is the amalgamation of all the stories that are sitting in front of me, if not one of them had been heard, then you would not be the personality that you are today.

[32.06s - 44.06s]
Hindi   : कुछ अलग होते कहानिया हम सब के लिए इतनी ज़रूरी इसले होती हैं. There are 7,139 languages in the world. I think there should be one more. The la

In [26]:
translated_all = translator.translate_segments(
    merged_transcript,
    source_language="hi",
    batch_size=8,
)

print("\nFull translation completed!")
print("Total segments:", len(translated_all))

Translated 8/90 segments
Translated 16/90 segments
Translated 24/90 segments
Translated 32/90 segments
Translated 40/90 segments
Translated 48/90 segments
Translated 56/90 segments
Translated 64/90 segments
Translated 72/90 segments
Translated 80/90 segments
Translated 88/90 segments
Translated 90/90 segments

Full translation completed!
Total segments: 90


In [27]:
import json

with open("translated_indic_module_test.json", "w", encoding="utf-8") as f:
    json.dump(
        translated_all,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved translated_indic_module_test.json")

Saved translated_indic_module_test.json


In [28]:
for segment in translated_all[:10]:
    print(f"\n[{segment['start']:.2f}s - {segment['end']:.2f}s]")
    print("Hindi   :", segment["text"])
    print("English :", segment["translated"])


[10.06s - 21.06s]
Hindi   : कहानिया हम सब की जिन्दिगी में बहुत महतुपून होती है, बहुत इंपूरेंट होती हैं। यह बात आप समझते हैं, जानते हैं। आप सोचीए वो सारी कहानिया जो हमने बचपं में स्कूलो में पढडली, हमारे माबाप नहीं हमें सुना दी।
English : Stories are very important, very important in all our lives. You understand, you know. You think of all the stories that we read in school as children, not our parents told us.

[21.06s - 32.06s]
Hindi   : हमारे दोस्तों से हमने कुछ सीखली वो सारी कहानियों का अमाल्गमेशन है शक्स जो मेरे सामने बैठा है उनमेसे एक भी कहानि अगर नहीं सुनाई गय होती तो आज आप जो पस्नालिटी है वो नहीं होते
English : Something we learned from our friends is the amalgamation of all the stories that are sitting in front of me, if not one of them had been heard, then you would not be the personality that you are today.

[32.06s - 44.06s]
Hindi   : कुछ अलग होते कहानिया हम सब के लिए इतनी ज़रूरी इसले होती हैं. There are 7,139 languages in the world. I think there should be one more. The la

In [29]:
for segment in translated_all[-5:]:
    print(f"\n[{segment['start']:.2f}s - {segment['end']:.2f}s]")
    print("Hindi   :", segment["text"])
    print("English :", segment["translated"])


[839.01s - 849.01s]
Hindi   : We have TEDx. जो आपका नजरीया आपकी कहानी लोगो तक पहुछाता है यह अपने आप में गौथ खुपसुरत कहानी है न? तो एक बार यह और कुशुते हूँ, तेडइक्स के लिए जोर से ताली बजाएंगे हम लोग यार।
English : We have TEDx. Your point of view brings your story to people. It's a cute story in itself, isn't it? So once again, I'll clap my hands out loud for TEDx, we guys.

[849.01s - 862.43s]
Hindi   : मैं वादा करता हूँ कि अपनी कहानी को नजरनदाज अगर आपने नहीं किया उसे लेकर बाहर आये तो तेडइक्स में या आपके आसपास वाले वो कहानिया बाहर लोगो तक पॉछा देंगे।
English : I promise that if you don't come out with your story, I'll get those stories out to people in Texas or around you.

[862.43s - 871.83s]
Hindi   : तो भाज के बाद अपणी कहानि को नद्रणदाजन्मत्ं करना, अपणी किटाब में जितना चाहिँ लिखते चले जाना और कुछ होणा हो, नंहीं चाहे हां 분들सारे किर्डार सो जाएं.
English : So, after the bhaj, you have to keep your story fresh, keep writing as much as you want in your kitab, and whatever happens, you d

In [30]:
%cd /content
!rm -rf Youtube-video-dubbing-system
!git clone https://github.com/harshita10sharma/Youtube-video-dubbing-system.git

/content
Cloning into 'Youtube-video-dubbing-system'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 90 (delta 46), reused 64 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 33.79 MiB | 27.79 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [31]:
%cd /content/Youtube-video-dubbing-system
!git log --oneline -5

/content/Youtube-video-dubbing-system
f85404d (HEAD -> main, origin/main, origin/HEAD) implemented indic translator and checked correct output as well
04645ab Add IndicTrans2 translation backend
05d4c0d created one more translator for indian specific languages
d3d94ff replacingprevious google translator with Indic translator
8040d25 converts translation from any specific language to english, instead of hindi forced


In [32]:
!mkdir -p data/transcripts
!cp /content/transcript.json data/transcripts/transcript.json

print("Transcript copied successfully!")

Transcript copied successfully!


In [33]:
import torch
import transformers
from IndicTransToolkit.processor import IndicProcessor

print("CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("IndicTransToolkit: OK")

CUDA: True
Transformers: 4.53.2
IndicTransToolkit: OK


In [36]:
!pip install -q deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.4 MB/s eta 0:00:00


In [37]:
from src.translator import translate_segments

import json

with open(
    "data/transcripts/transcript.json",
    "r",
    encoding="utf-8"
) as f:
    transcript = json.load(f)

print("Original Whisper segments:", len(transcript))

translated = translate_segments(
    transcript,
    source_language="hi"
)

print("\n================================")
print("INTEGRATION TEST COMPLETED")
print("Translated chunks:", len(translated))
print("================================")

Original Whisper segments: 260
[14:33:32] Merged 260 Whisper segments into 90 translation chunks.
[14:33:32] Translating 90 segments (hi -> en)...
      [1/90] Stories are very important and very influential in the lives of all of us. You understand and know this. Just think about all those stories which we read in schools in our childhood, but it was not our parents who told them to us.
      [2/90] What we learned from our friends is the amalgamation of all the stories. If even one of the stories of the person sitting in front of me had not been told, then you would not have become the personality you are today.
      [3/90] This is why different stories are so important to all of us. There are 7,139 languages ​​in the world. I think there should be one more. The language of storytelling.
      [4/90] The best way to communicate and communicate. Imagine. Imagine that you are in a ship. There is complete darkness.
      [5/90] Storms are coming everywhere. Very big waves are rising an